# Phase 3: Embedding & FAISS Indexing

**Pipeline**: Vietnamese Financial News RAG System — v3  
**Hardware**: ⚡ Google Colab T4 GPU (REQUIRED)  
**Model**: `intfloat/multilingual-e5-large` (560M params · dim=1024)

**Parallelism**: Run this notebook on 3 separate Colab accounts simultaneously.  
Change `TARGET_STRATEGY` below to match the account assignment:

| Account | TARGET_STRATEGY |
|---------|-----------------|
| Account 1 | `fixed_size` |
| Account 2 | `sentence_aware` |
| Account 3 | `article_level` |

All cells are **idempotent** — safe to re-run after a Colab timeout.  
On resume, the checkpoint is detected automatically and encoding continues from where it stopped.

## Cell 0 — Environment Setup & GPU Verification

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path

# ── Strategy selector (change this per Colab account) ──────────────────────
TARGET_STRATEGY = 'fixed_size'   # options: 'fixed_size' | 'sentence_aware' | 'article_level'

# ── Environment detection ───────────────────────────────────────────────────
def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

IN_COLAB = is_colab()

# ── GPU check ──────────────────────────────────────────────────────────────
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Environment Setup ──────────────────────────────────────────────────────
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    GIT_DIR = Path('/content/rag-vn-finance')
    REPO_ROOT = GIT_DIR 
    DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/rag-vn-finance/implementation')
    
    # Xóa triệt để repo cũ để tránh lỗi xung đột (conflict) từ Colab auto-save
    print("Đang dọn dẹp không gian tạm và tải mã nguồn mới nhất từ GitHub...")
    if GIT_DIR.exists():
        shutil.rmtree(GIT_DIR, ignore_errors=True)
        
    # Clone mới hoàn toàn
    result = subprocess.run(['git', 'clone', 'https://github.com/thong7d/rag-vn-finance.git', str(GIT_DIR)], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Lỗi khi clone git:\n{result.stderr}")
    
    # Install dependencies
    req_path = REPO_ROOT / 'requirements.txt'
    if req_path.exists():
        os.system(f'pip install -r "{req_path}" -q')
        if device == 'cuda':
            os.system('pip install faiss-gpu -q')
else:
    REPO_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
    DRIVE_DATA_ROOT = REPO_ROOT

# Kiểm tra an toàn trước khi nạp sys.path
src_path = REPO_ROOT / 'src'
if not src_path.exists():
    raise FileNotFoundError(
        f"❌ LỖI: Không tìm thấy thư mục src tại {src_path}.\n"
        f"HÃY KIỂM TRA LẠI GITHUB: Đảm bảo bạn đã push thư mục 'src' lên repository (nhánh main)!"
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Target strategy: {TARGET_STRATEGY}")
print(f"Runtime: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Device: {device}")
print(f"Code Root (Git): {REPO_ROOT}")
print(f"Data Root (Drive): {DRIVE_DATA_ROOT}")
print("\n✅ Cell 0 complete. Hệ thống đã nhận diện được module 'src'.")

## Cell 1 — Load Config & Resolve Paths

In [ ]:
import json
import pandas as pd
from src.utils import load_config, resolve_path

config = load_config(REPO_ROOT / 'configs' / 'config.yaml')
emb_cfg = config['embedding']

# ── Resolve paths (SỬ DỤNG DRIVE_DATA_ROOT CHO DATA) ──────────────────────
chunks_base = resolve_path(config['chunking'], 'output_dir')
if not os.path.isabs(chunks_base):
    chunks_base = str(DRIVE_DATA_ROOT / chunks_base)

emb_base = resolve_path(emb_cfg, 'output_dir')
if not os.path.isabs(emb_base):
    emb_base = str(DRIVE_DATA_ROOT / emb_base)

idx_base = resolve_path(config['indexing'], 'output_dir')
if not os.path.isabs(idx_base):
    idx_base = str(DRIVE_DATA_ROOT / idx_base)

# Per-strategy paths
CHUNKS_PATH     = os.path.join(chunks_base, TARGET_STRATEGY, 'chunks.parquet')
EMB_DIR         = os.path.join(emb_base,    TARGET_STRATEGY)
EMB_NPY_PATH    = os.path.join(EMB_DIR,     'embeddings.npy')
CHECKPOINT_PATH = os.path.join(EMB_DIR,     'checkpoint.json')
IDX_DIR         = os.path.join(idx_base,    TARGET_STRATEGY)
FAISS_PATH      = os.path.join(IDX_DIR,     'index.faiss')

# Tạo folder nếu chưa có trên Drive
os.makedirs(EMB_DIR, exist_ok=True)
os.makedirs(IDX_DIR, exist_ok=True)

print(f"Strategy         : {TARGET_STRATEGY}")
print(f"Chunks input     : {CHUNKS_PATH}")
print(f"Embeddings dir   : {EMB_DIR}")
print(f"Embeddings .npy  : {EMB_NPY_PATH}")
print(f"Checkpoint       : {CHECKPOINT_PATH}")
print(f"FAISS index      : {FAISS_PATH}")

assert os.path.exists(CHUNKS_PATH), f"❌ Chunks file not found: {CHUNKS_PATH}"
print("\n✅ Paths resolved. Cell 1 complete.")

## Cell 2 — Load Chunk Dataset

In [ ]:
df_chunks = pd.read_parquet(CHUNKS_PATH)
print(f"Chunks loaded: {len(df_chunks):,} rows")
print(f"Columns      : {df_chunks.columns.tolist()}")
print(f"Sample chunk_id: {df_chunks['chunk_id'].iloc[0]}")

# Verify required columns
required = ['chunk_id', 'doc_id', 'text', 'strategy',
            'source', 'category', 'year', 'title', 'url',
            'tickers', 'is_historical', 'numerical_density', 'entities']
missing = [c for c in required if c not in df_chunks.columns]
if missing:
    raise ValueError(f"Missing columns in chunks: {missing}")

print(f"\n✅ Schema verified. {len(df_chunks):,} chunks ready.")
df_chunks.head(2)

## Cell 3 — Apply "passage: " Prefix

> **Required by intfloat/multilingual-e5-large**  
> Documents must be prefixed with `"passage: "` and queries with `"query: "`.  
> Missing this prefix significantly degrades retrieval quality.

In [ ]:
from src.embedding import PASSAGE_PREFIX

# Apply prefix to every chunk text
prefixed_texts = [PASSAGE_PREFIX + str(t) for t in df_chunks['text'].tolist()]

print(f"Total texts to embed : {len(prefixed_texts):,}")
print(f"Prefix applied       : '{PASSAGE_PREFIX}'")
print(f"Sample (first 80 chars): {prefixed_texts[0][:80]}...")

print("\nCell 3 complete.")

## Cell 4 — Load Embedding Model

In [ ]:
from sentence_transformers import SentenceTransformer

model_name = emb_cfg['model_name']
print(f"Loading model: {model_name}")
print("(First run downloads ~2.2 GB — subsequent runs load from cache)")

embedding_model = SentenceTransformer(model_name, device=device)
embedding_model.max_seq_length = 512

print(f"\n✅ Model loaded on {device}")
print(f"   Embedding dimension : {embedding_model.get_sentence_embedding_dimension()}")
print(f"   Max sequence length : {embedding_model.max_seq_length}")

## Cell 5 — Encode Chunks (with Checkpointing)

> ⏱️ **TIME WARNING**: ~2–3 hours on T4 for fixed_size (~42K chunks).  
> If the session times out, just re-run this cell — it resumes from the checkpoint automatically.

In [ ]:
from src.embedding import encode_chunks_with_checkpoint

batch_size       = emb_cfg.get('batch_size', 64)
checkpoint_every = emb_cfg.get('checkpoint_every', 100)

print(f"Batch size      : {batch_size}")
print(f"Checkpoint every: {checkpoint_every} batches")
print(f"Total chunks    : {len(prefixed_texts):,}")
print(f"Total batches   : {(len(prefixed_texts) + batch_size - 1) // batch_size:,}")

if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        ckpt = json.load(f)
    print(f"\n🔄 Resuming — {ckpt.get('completed_chunks', 0)}/{len(prefixed_texts)} chunks done")
else:
    print("\n🚀 Starting fresh encoding run...")

embeddings_mmap = encode_chunks_with_checkpoint(
    texts=prefixed_texts,
    model=embedding_model,
    output_npy_path=EMB_NPY_PATH,
    checkpoint_path=CHECKPOINT_PATH,
    batch_size=batch_size,
    checkpoint_every=checkpoint_every,
)

print(f"\n✅ Encoding complete. Embedding matrix shape: {embeddings_mmap.shape}")

## Cell 6 — Build & Save FAISS Index

In [ ]:
from src.embedding import build_faiss_index

if os.path.exists(FAISS_PATH):
    print(f"FAISS index already exists: {FAISS_PATH}")
    print("Loading existing index...")
    import faiss
    index = faiss.read_index(FAISS_PATH)
    print(f"✅ Loaded — {index.ntotal:,} vectors")
else:
    print("Building FAISS index...")
    index = build_faiss_index(
        npy_path=EMB_NPY_PATH,
        index_output_path=FAISS_PATH,
    )
    print(f"\n✅ FAISS index built and saved — {index.ntotal:,} vectors")

print(f"Index type : {type(index).__name__}")
print(f"Dimension  : {index.d}")

## Cell 7 — Align & Save Metadata

In [ ]:
from src.embedding import align_and_save_metadata

meta_path    = os.path.join(IDX_DIR, 'metadata.parquet')
ids_path     = os.path.join(IDX_DIR, 'chunk_ids.json')

if os.path.exists(meta_path) and os.path.exists(ids_path):
    print(f"Metadata files already exist:")
    print(f"  {meta_path}")
    print(f"  {ids_path}")
else:
    ids_path, meta_path = align_and_save_metadata(
        df_chunks=df_chunks,
        output_dir=IDX_DIR,
    )

print("\nVerifying alignment...")
import faiss
idx = faiss.read_index(FAISS_PATH)
df_meta = pd.read_parquet(meta_path)
with open(ids_path) as f:
    ids = json.load(f)

assert idx.ntotal == len(df_meta) == len(ids), (
    f"Alignment mismatch: index={idx.ntotal}  meta={len(df_meta)}  ids={len(ids)}"
)
print(f"✅ Alignment verified — {idx.ntotal:,} vectors | {len(df_meta):,} metadata rows")

## Cell 8 — Smoke Test: Query the Index

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Reload model & index (handles session resets cleanly)
if 'embedding_model' not in dir() or embedding_model is None:
    embedding_model = SentenceTransformer(emb_cfg['model_name'], device=device)

idx      = faiss.read_index(FAISS_PATH)
df_meta  = pd.read_parquet(meta_path)

# Test query with the mandatory "query: " prefix
TEST_QUERY = "query: Lãi suất ngân hàng Việt Nam năm 2023"

query_emb = embedding_model.encode(TEST_QUERY, normalize_embeddings=True).reshape(1, -1).astype('float32')
scores, faiss_ids = idx.search(query_emb, 5)

print(f"Query: {TEST_QUERY}")
print(f"\nTop-5 results:")
for rank, (fid, score) in enumerate(zip(faiss_ids[0], scores[0]), 1):
    row = df_meta.iloc[fid]
    print(f"  #{rank}  score={score:.4f}  chunk_id={row['chunk_id']}")
    print(f"       title  : {row['title'][:70]}...")
    print(f"       year   : {row['year']}  source: {row['source']}")

print("\n✅ Smoke test passed.")

## Cell 9 — Cleanup Temporary .npy File

> **Run this cell ONLY after verifying the FAISS index is correct in Cell 8.**  
> The intermediate `.npy` embedding file is no longer needed once the FAISS index is built.  
> Deleting it frees ~200–400 MB of Drive space per strategy.

In [ ]:
# Safety guard: only delete if FAISS index exists and is non-empty
import faiss

idx_check = faiss.read_index(FAISS_PATH)
assert idx_check.ntotal > 0, "FAISS index is empty — do NOT delete the .npy file!"

if os.path.exists(EMB_NPY_PATH):
    npy_size_mb = os.path.getsize(EMB_NPY_PATH) / 1e6
    os.remove(EMB_NPY_PATH)
    print(f"✅ Deleted: {EMB_NPY_PATH}  ({npy_size_mb:.0f} MB freed)")
else:
    print(f"ℹ️  File already removed: {EMB_NPY_PATH}")

print(f"\nFinal outputs for strategy '{TARGET_STRATEGY}':")
for f in [FAISS_PATH, meta_path, ids_path, CHECKPOINT_PATH]:
    size = os.path.getsize(f) / 1e6 if os.path.exists(f) else 0
    print(f"  {'✅' if os.path.exists(f) else '❌'}  {f}  ({size:.1f} MB)")

print(f"\n✅ Phase 3 complete for strategy: {TARGET_STRATEGY}")
print("Confirm 'Xong' before proceeding to Phase 4 (BM25 Indexing).")